# 14. Identifying and Removing Duplicate Records

This notebook demonstrates a complete deduplication workflow in Pandas.

Learning goals:
- Understand exact vs partial duplicates
- Detect duplicate rows in a DataFrame
- Remove duplicates intentionally
- Verify cleanup results

In [1]:
from pathlib import Path

import pandas as pd


PROJECT_ROOT = Path.cwd().resolve().parent
CSV_PATH = PROJECT_ROOT / "data" / "raw" / "employee_survey_2026_Q1.csv"
KEY_COLUMNS = ["employee_id", "survey_date"]

print(f"Dataset path: {CSV_PATH}")

Dataset path: C:\Users\USER\OneDrive\Desktop\Data-science\S86_Trivin_Insight_Engine_Sprint3\data\raw\employee_survey_2026_Q1.csv


## 1. Load and Inspect the Base Dataset

We start by loading the survey data and confirming whether it already contains duplicates.

In [2]:
df = pd.read_csv(CSV_PATH)

print(f"Shape: {df.shape}")
print(f"Full-row duplicates: {df.duplicated().sum()}")
print(f"Key-based duplicates ({KEY_COLUMNS}): {df.duplicated(subset=KEY_COLUMNS).sum()}")

df.head()

Shape: (30, 9)
Full-row duplicates: 0
Key-based duplicates (['employee_id', 'survey_date']): 0


,employee_id,department,survey_date,satisfaction_score,work_life_balance,management_support,career_growth,team_collaboration,response_text
0,1001,Engineering,2026-01-15,7,8,6,5,9,Great team environment but limited growth oppo...
1,1002,Marketing,2026-01-16,9,9,8,7,8,Very satisfied with work-life balance and supp...
2,1003,Sales,2026-01-17,4,3,4,3,5,High pressure environment with poor work-life ...
3,1004,Engineering,2026-01-18,8,7,8,7,9,Good technical challenges and collaborative team
4,1005,HR,2026-01-19,6,7,5,4,6,Decent work environment but lack career advanc...


## 2. Build a Demo DataFrame with Duplicates

The raw data currently has no duplicates, so we intentionally add:
- Exact duplicates (all values match)
- A partial duplicate (same key, changed text)

This lets us practice detection and cleanup safely.

In [3]:
exact_duplicates = df.iloc[[1, 7]].copy()

partial_duplicate = df.iloc[[3]].copy()
partial_duplicate.loc[:, "response_text"] = (
    partial_duplicate["response_text"]
    .astype(str)
    .str.replace("workload", "heavy workload", regex=False)
)

demo_df = pd.concat([df, exact_duplicates, partial_duplicate], ignore_index=True)

print(f"Original shape: {df.shape}")
print(f"Demo shape: {demo_df.shape}")
demo_df.tail(5)

Original shape: (30, 9)
Demo shape: (33, 9)


,employee_id,department,survey_date,satisfaction_score,work_life_balance,management_support,career_growth,team_collaboration,response_text
28,1029,Finance,2026-02-13,8,7,8,7,8,Professional environment with growth potential
29,1030,HR,2026-02-14,6,6,5,5,6,Consistent policies but slow adaptation to new...
30,1002,Marketing,2026-01-16,9,9,8,7,8,Very satisfied with work-life balance and supp...
31,1008,Sales,2026-01-22,3,2,3,2,4,Unrealistic targets and constant pressure from...
32,1004,Engineering,2026-01-18,8,7,8,7,9,Good technical challenges and collaborative team


## 3. Detect Duplicate Rows

We use two duplicate checks:
- `duplicated()` for exact full-row duplicates
- `duplicated(subset=...)` for key-level duplicates

In [4]:
full_mask = demo_df.duplicated(keep=False)
key_mask = demo_df.duplicated(subset=KEY_COLUMNS, keep=False)

print(f"Full-row duplicate count: {demo_df.duplicated().sum()}")
print(f"Key-based duplicate count: {demo_df.duplicated(subset=KEY_COLUMNS).sum()}")

print("\nRows involved in full-row duplicates:")
display(demo_df.loc[full_mask].sort_values(KEY_COLUMNS))

print("\nRows involved in key-based duplicates:")
display(demo_df.loc[key_mask].sort_values(KEY_COLUMNS))

Full-row duplicate count: 3
Key-based duplicate count: 3

Rows involved in full-row duplicates:


,employee_id,department,survey_date,satisfaction_score,work_life_balance,management_support,career_growth,team_collaboration,response_text
1,1002,Marketing,2026-01-16,9,9,8,7,8,Very satisfied with work-life balance and supp...
30,1002,Marketing,2026-01-16,9,9,8,7,8,Very satisfied with work-life balance and supp...
3,1004,Engineering,2026-01-18,8,7,8,7,9,Good technical challenges and collaborative team
32,1004,Engineering,2026-01-18,8,7,8,7,9,Good technical challenges and collaborative team
7,1008,Sales,2026-01-22,3,2,3,2,4,Unrealistic targets and constant pressure from...
31,1008,Sales,2026-01-22,3,2,3,2,4,Unrealistic targets and constant pressure from...



Rows involved in key-based duplicates:


,employee_id,department,survey_date,satisfaction_score,work_life_balance,management_support,career_growth,team_collaboration,response_text
1,1002,Marketing,2026-01-16,9,9,8,7,8,Very satisfied with work-life balance and supp...
30,1002,Marketing,2026-01-16,9,9,8,7,8,Very satisfied with work-life balance and supp...
3,1004,Engineering,2026-01-18,8,7,8,7,9,Good technical challenges and collaborative team
32,1004,Engineering,2026-01-18,8,7,8,7,9,Good technical challenges and collaborative team
7,1008,Sales,2026-01-22,3,2,3,2,4,Unrealistic targets and constant pressure from...
31,1008,Sales,2026-01-22,3,2,3,2,4,Unrealistic targets and constant pressure from...


## 4. Remove Duplicates Intentionally

We apply deduplication in two steps:
1. Remove exact duplicates first (`keep='first'`)
2. Remove key-based duplicates on `employee_id + survey_date` (`keep='last'`)

In [5]:
dedup_full_df = demo_df.drop_duplicates(keep="first")
dedup_final_df = dedup_full_df.drop_duplicates(subset=KEY_COLUMNS, keep="last")

print(f"After full-row dedup: {dedup_full_df.shape}")
print(f"After key-based dedup: {dedup_final_df.shape}")

dedup_final_df.tail(5)

After full-row dedup: (30, 9)
After key-based dedup: (30, 9)


,employee_id,department,survey_date,satisfaction_score,work_life_balance,management_support,career_growth,team_collaboration,response_text
25,1026,Sales,2026-02-10,6,5,5,5,6,Challenging but manageable sales environment
26,1027,Engineering,2026-02-11,9,9,8,8,10,Innovative projects and supportive management
27,1028,IT,2026-02-12,5,6,4,4,5,Lack of resources and understaffed team
28,1029,Finance,2026-02-13,8,7,8,7,8,Professional environment with growth potential
29,1030,HR,2026-02-14,6,6,5,5,6,Consistent policies but slow adaptation to new...


## 5. Verify Deduplication Results

Always verify:
- Shape before vs after
- Remaining duplicate counts
- Key uniqueness integrity

In [6]:
print("Shape comparison:")
print(f"Before deduplication: {demo_df.shape}")
print(f"After deduplication:  {dedup_final_df.shape}")

remaining_full = dedup_final_df.duplicated().sum()
remaining_key = dedup_final_df.duplicated(subset=KEY_COLUMNS).sum()

print("\nRemaining duplicates:")
print(f"Full-row duplicates: {remaining_full}")
print(f"Key-based duplicates: {remaining_key}")

before_unique_keys = demo_df[KEY_COLUMNS].drop_duplicates().shape[0]
after_unique_keys = dedup_final_df[KEY_COLUMNS].drop_duplicates().shape[0]

print("\nKey integrity:")
print(f"Unique keys before cleanup: {before_unique_keys}")
print(f"Unique keys after cleanup:  {after_unique_keys}")

assert remaining_full == 0, "Full-row duplicates remain after cleanup"
assert remaining_key == 0, "Key-based duplicates remain after cleanup"
assert after_unique_keys == before_unique_keys, "Key integrity changed unexpectedly"

print("\nVerification passed.")

Shape comparison:
Before deduplication: (33, 9)
After deduplication:  (30, 9)

Remaining duplicates:
Full-row duplicates: 0
Key-based duplicates: 0

Key integrity:
Unique keys before cleanup: 30
Unique keys after cleanup:  30

Verification passed.


## Key Takeaways

- Always inspect duplicates before removing rows.
- Use `subset=` for business-key deduplication.
- The `keep` strategy should be explainable (`first` vs `last`).
- Verification is mandatory after cleanup.